In [ ]:
import torch
from torch_geometric.loader import DataLoader
from model import WDMPNNModel   # 你之前写的 model.py
from train import train_one_epoch, evaluate, WMAELoss, compute_task_stats
import pandas as pd

# ------------------ 构造最小数据集 ------------------
# 假设你 Kaggle 预处理过的数据在 train_filtered 里
from data import build_pyg_dataset, get_train_test, add_extra_data, clean_smiles, filter_train_data, replace_all_R_with_C

In [ ]:
train, test, sub = get_train_test()
train_added = add_extra_data(train)
train_added.rename(columns={"SMILES": "SMILES_raw"}, inplace=True)
train_added["SMILES"] = train_added["SMILES_raw"].apply(replace_all_R_with_C)
train_cleaned = clean_smiles(train_added)
train_filtered = filter_train_data(train_cleaned)

In [ ]:
tasks = ["Tg", "Tc", "Density"]
ds = build_pyg_dataset(
    train_filtered["SMILES"],
    targets=train_filtered[tasks].values,
    cache_path="train_small.pkl",
)

loader = DataLoader(ds, batch_size=8, shuffle=True)

In [ ]:
for batch in loader:
    print("y shape:", batch.y.shape)
    print(batch.y[:5])  # 看前5行
    break

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
sample = ds[0]

model = WDMPNNModel(
    node_dim=sample.x.size(-1),
    edge_dim=sample.edge_attr.size(-1),
    hidden_dim=64,          # 小点，便于测试
    num_layers=2,
    tasks=tasks,
    pre_node_dim=32,        # 假装预训练时维度不同
    pre_edge_dim=16,
    adapter_kind="linear",
).to(device)

In [ ]:
print(device)

In [ ]:
n_dict, r_dict = compute_task_stats(train_filtered.iloc[:20], tasks)
loss_fn = WMAELoss(tasks, n_dict, r_dict)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
train_loss, train_task_loss = train_one_epoch(model, loader, optimizer, loss_fn, device, tasks)
print("Train loss:", train_loss)
print("Per-task loss:", train_task_loss)